In [ ]:
!pip install -r requirements.txt

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/mnt/share_data_78/howard/models/qwen3-0.6b",
    max_seq_length = 1024,
    full_finetuning = True,
    qat_scheme = "phone-deployment", # Flag for phone deployment
)

model.save_pretrained_torchao("phone_model", tokenizer = tokenizer)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp2/howard/venv_manager/llm_on_iphone/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: You selected full finetuning support, but 4bit / 8bit is enabled - disabling LoRA / QLoRA.
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 2. Max memory: 23.518 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained
/mnt/share_data_78/howard/models/qwen3-0.6b does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Unsloth: Applying QAT to mitigate quantization degradation


In [2]:
!python -m executorch.examples.models.qwen3.convert_weights "phone_model" pytorch_model_converted.bin

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
I tokenizers:regex.cpp:27] Registering override fallback regex
<frozen runpy>:128: RuntimeWarning: 'executorch.examples.models.qwen3.convert_weights' found in sys.modules after import of package 'executorch.examples.models.qwen3', but prior to execution of 'executorch.examples.models.qwen3.convert_weights'; this may result in unpredictable behaviour
Loading checkpoint...
Loading checkpoint from pytorch_model directory
Converting checkpoint...
Saving checkpoint...
Done.


In [3]:
!curl -L -o 0.6B_config.json https://raw.githubusercontent.com/pytorch/executorch/main/examples/models/qwen3/config/0_6b_config.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   348  100   348    0     0    812      0 --:--:-- --:--:-- --:--:--   813


In [4]:
# Export to ExecuTorch pte file
!python -m executorch.examples.models.llama.export_llama \
    --model "qwen3_0_6b" \
    --checkpoint pytorch_model_converted.bin \
    --params 0.6B_config.json \
    --output_name qwen3_0.6B_model.pte \
    -kv --use_sdpa_with_kv_cache -X --xnnpack-extended-ops \
    --max_context_length 1024 --max_seq_length 128 --dtype fp32 \
    --metadata '{"get_bos_id":199999, "get_eos_ids":[200020,199999]}'

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
I tokenizers:regex.cpp:27] Registering override fallback regex
[INFO 2026-03-12 14:08:14,116 export_llama_lib.py:781] Applying quantizers: []
[INFO 2026-03-12 14:08:14,225 export_llama_lib.py:703] Checkpoint dtype: torch.bfloat16
[INFO 2026-03-12 14:08:14,225 custom_kv_cache.py:367] Replacing KVCache with CustomKVCache. This modifies the model in place.
[INFO 2026-03-12 14:08:14,250 custom_ops.py:34] Looking for libcustom_ops_aot_lib.so in /tmp2/howard/venv_manager/llm_on_iphone/lib/python3.12/site-packages/executorch/extension/llm/custom_ops
[INFO 2026-03-12 14:08:14,250 custom_ops.py:39] Loading custom ops library: /tmp2/howard/venv_manager/llm_on_iphone/lib/python3.12/site-packages/executorch/extension/llm/custom_ops/libcustom_ops_aot_lib.so
[INFO 2026-03-12 14:08:14,255 builder.py:192] Model after source transforms: Transform